In [1]:
import pandas as pd
import numpy as np

panel = pd.read_csv("../data/processed/panel_gdp.csv")
main = panel[panel["main_sample"]].dropna(subset=["s_may", "e_2025"])

both = main.dropna(subset=["s_apr"])
print(len(both), np.corrcoef(np.log(both["s_may"]), np.log(both["s_apr"]))[0, 1].round(3))

455 0.984


In [2]:
emp = pd.read_csv("../data/processed/employment_shares.csv")
raw = pd.read_csv("../data/raw/anthropic/aei_claude_ai_2026-06-26.csv",
                  usecols=["geo_id", "geo_level", "category_name", "metric_id", "date_start", "value"])
codes = {"AUT": "AT", "BEL": "BE", "CZE": "CZ", "DNK": "DK", "FIN": "FI", "FRA": "FR",
         "DEU": "DE", "GRC": "EL", "IRL": "IE", "ITA": "IT", "NLD": "NL", "POL": "PL",
         "PRT": "PT", "ROU": "RO", "ESP": "ES", "SWE": "SE",
         "BGR": "BG", "HUN": "HU", "LTU": "LT", "SVK": "SK"}

In [3]:
U = raw[(raw["geo_level"] == "country") & (raw["category_name"] == "overall") &
        (raw["metric_id"] == "usage_pct") & (raw["date_start"] == "2026-05-01")].copy()
U["geo"] = U["geo_id"].map(codes)
U = U.dropna(subset=["geo"]).set_index("geo")["value"].rename("U")
E = emp[emp["year"] == 2025].groupby("geo")["value"].sum().rename("E")
tot = pd.concat([U, E], axis=1, join="inner").loc[main["geo"].unique()]
tot["A"] = (tot["U"] / tot["E"]) / (tot["U"].sum() / tot["E"].sum())
tot.sort_values("A").round(3)

,U,E,A
geo,,,
PL,0.75,16951.3,0.522
RO,0.40,7651.6,0.617
EL,0.23,4272.6,0.635
CZ,0.30,5233.3,0.676
IT,1.50,23873.6,0.741
FI,0.18,2562.2,0.829
DE,2.98,42177.3,0.834
AT,0.37,4488.3,0.973
SE,0.46,5297.6,1.025


In [4]:
n = main.groupby("isco2")["geo"].nunique()
common = n[n == 16].index
cells = main[main["isco2"].isin(common)][["geo", "isco2", "s_may", "e_2025"]]
cells.columns = ["geo", "occ", "s", "e"]
other = cells.groupby("geo")[["s", "e"]].sum().rsub(1).reset_index().assign(occ="other")
t = pd.concat([cells, other]).merge(tot, left_on="geo", right_index=True)
print(len(common), t.groupby("geo")[["s", "e"]].sum().round(6).drop_duplicates())

26        s    e
geo          
AT   1.0  1.0


In [5]:
t["Uw"] = t["U"] * t["s"]
t["Ew"] = t["E"] * t["e"]
star = t.groupby("occ")[["Uw", "Ew"]].sum()
star["s_star"] = star["Uw"] / tot["U"].sum()
star["e_star"] = star["Ew"] / tot["E"].sum()
t = t.merge(star[["s_star", "e_star"]], left_on="occ", right_index=True)

t["a"] = t["A"] * t["s"] / t["e"]
t["a_star"] = t["s_star"] / t["e_star"]
t["comp"] = (t["e"] - t["e_star"]) * (t["a"] + t["a_star"]) / 2
t["inten"] = (t["e"] + t["e_star"]) / 2 * (t["a"] - t["a_star"])

res = t.groupby("geo").agg(A=("A", "first"), comp=("comp", "sum"), inten=("inten", "sum"))
res["gap"] = res["A"] - 1
print((res["gap"] - res["comp"] - res["inten"]).abs().max())
res.sort_values("gap").round(3)

3.0531133177191805e-16


,A,comp,inten,gap
geo,,,,
PL,0.522,-0.077,-0.401,-0.478
RO,0.617,-0.285,-0.098,-0.383
EL,0.635,-0.201,-0.164,-0.365
CZ,0.676,-0.040,-0.283,-0.324
IT,0.741,-0.241,-0.018,-0.259
FI,0.829,0.039,-0.210,-0.171
DE,0.834,0.000,-0.167,-0.166
AT,0.973,0.007,-0.034,-0.027
SE,1.025,0.122,-0.097,0.025


In [6]:
v = res["gap"].var()
print("composition:", round(res["gap"].cov(res["comp"]) / v, 3))
print("intensity:", round(res["gap"].cov(res["inten"]) / v, 3))
res.to_csv("../output/tables/decomposition.csv")

composition: 0.226
intensity: 0.774
